# xAI Grok 4.6 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

xAI's [Grok 4.6](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-xai-grok-4-6.html) is built for coding, agentic work and long-running agents, with a
500K-token context window. It reasons on every request, its reasoning is encrypted, and
it reads images. It is on both endpoints, and each wants it addressed differently, as
the table shows. Grok 4.3 has its own notebook, [`01-grok-4-3`](01-grok-4-3.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Grok 4.6 | `xai.grok-4.6` | `us.xai.grok-4.6` or `global.xai.grok-4.6` | `bedrock-mantle` in `us-west-2`, and `us-gov-east-1` per the card; `bedrock-runtime` through an inference profile |

IDs, Regions and lifecycle read from the model card on 20 October 2026. Its
Programmatic Access table gives `bedrock-runtime` the model ID `xai.grok-4.6` with `Not
supported` as the In-Region endpoint URL, so that endpoint reaches the model only
through a cross-Region inference profile: `us.xai.grok-4.6` from the six source Regions
the card's US geo table lists (`us-east-1`, `us-east-2`, `us-west-1`, `us-west-2`,
`ca-central-1` and `ca-west-1`, and that profile answered in the Canadian ones on 18
October 2026), or `global.xai.grok-4.6`, which the card marks available in every Region
on its `bedrock-runtime` list except the GovCloud pair. `bedrock-mantle` takes the bare
ID, In-Region in `us-west-2` and `us-gov-east-1` (GovCloud East), which this notebook
does not call (section 11). The card marks Responses, Chat Completions, Converse and
Invoke all supported, and reads Active with `Model EOL date: N/A` and an "EOL no sooner
than" date of 18 August 2027, under the [policy for models launched before 7 September
2026](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle-legacy.html).

This notebook calls the **Responses** API on `bedrock-runtime`, with the OpenAI SDK at
`https://bedrock-runtime.{region}.amazonaws.com/openai/v1`. Chat Completions,
Converse and InvokeModel are on the same endpoint (sections 8, 9 and 10), and section 11
calls the model on `bedrock-mantle`.

**Covered:** first call, streaming, multi-turn and server-side state, reasoning, tool
use, structured output, vision, Chat Completions, Converse, InvokeModel,
`bedrock-mantle` with background mode, IAM permissions, prompt caching, and service
tiers.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes a lenient JSON
parser, a print that redacts account IDs, and the sample slide.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import re
import sys

import boto3
from aws_bedrock_token_generator import provide_token
import openai
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import parse_json_lenient, safe_print

REGION = "us-east-1"
GROK = "us.xai.grok-4.6"  # the US geographic inference profile

# bedrock-runtime's OpenAI-compatible path takes the same short-term Bedrock API key as
# bedrock-mantle. Reasoning calls can take minutes, so set a client timeout.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=300,
)
bedrock = boto3.client("bedrock", region_name=REGION)


def answer_text(response) -> str:
    """The answer, without the \\confidence{N} marker Grok 4.6 sometimes ends it with."""
    return re.sub(r"\s*\\confidence\{\d+\}\s*$", "", response.output_text or "").strip()


def says_five_cents(text: str) -> bool:
    """True when the answer names five cents, as $0.05, .05, 5 cents or five cents."""
    amount = text.lower().replace("$", "").replace(",", "")
    # Lookarounds rather than \b, which does not match before the dot in ".05", and
    # which would let $1.05 or 0.055 through.
    return bool(re.search(r"(?<![\d.])0?\.05(?!\d)", amount)
                or re.search(r"\b(5|five)\s*cents?\b", amount))

## 1. First call

On `bedrock-runtime` name an inference profile as the model; the bare ID returns 400
`Invocation of model ID xai.grok-4.6 with on-demand throughput isn’t supported`. Grok
4.6 reasons before it answers and spends output tokens doing it, so a small
`max_output_tokens` returns HTTP 200 with `status` `incomplete` and no text; a budget of
32 did that in September 2026 runs, and the cells here give it 2,000. In the same runs
it sometimes ended an answer with a `\confidence{100}` marker, which `answer_text` in
the setup cell removes.

Sampling does not use the OpenAI defaults. A request that sends neither parameter comes
back echoing `temperature` 0.7 and `top_p` 0.95, where the OpenAI API's own default for
both is 1, so set them explicitly if your output depends on them. The accepted ranges are
the OpenAI ones, `temperature` to 2 and `top_p` to 1, and the top of the temperature range
is not usable on a model that reasons on every request: at `temperature` 2.0 none of eight
trials answered, seven spending the whole 2,000-token budget on reasoning and one coming
back with `status` `completed` and empty text. Keep `temperature` at or below 1.5, which
answered in four trials of four, and check that the text is non-empty as well as that
`status` is `completed` (October 2026).

Every call in this notebook passes `store=False`. The Responses API stores each response
for 30 days by default, and AWS advises passing `store=False` on every request you do
not want retained. With `global.`, a stored response can be kept in any commercial
Region the profile routes to, so use `us.` when data residency matters.

In [2]:
response = client.responses.create(
    model=GROK,
    input="What is the capital of Australia? Answer in one word.",
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(response)
print(answer)
# Reasoning is on at `high` by default here, so the one-word answer can arrive with a
# sentence after it: look for the word rather than comparing the whole string. A wrong
# `status` says the call finished, and says nothing about whether the answer is right.
print("says Canberra:", "canberra" in answer.lower())
print(f"status={response.status}  output_tokens={response.usage.output_tokens}")
if response.status != "completed":
    print("incomplete:", getattr(response.incomplete_details, "reason", None))

Canberra
says Canberra: True
status=completed  output_tokens=140


## 2. Streaming

With `stream=True` the SDK yields typed events, and answer text arrives in
`response.output_text.delta` events. The reasoning is encrypted, so no reasoning deltas
arrive. The `\confidence{100}` marker from section 1 can arrive in the last deltas, so a
UI that shows deltas as they arrive shows it too. The cell prints the raw deltas, then
the assembled answer from the final event with the marker removed by `answer_text`.

In [3]:
stream = client.responses.create(
    model=GROK,
    input="Count from 1 to 10, separated by commas.",
    max_output_tokens=2000,
    store=False,
    stream=True,
)
reasoning_deltas = 0
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type in ("response.reasoning.delta", "response.reasoning_text.delta"):
        reasoning_deltas += 1  # count them here; a UI could show them as they arrive
    elif event.type in ("response.completed", "response.incomplete"):
        answer = answer_text(event.response)  # the assembled text, marker removed
        print(f"\nstatus={event.response.status}  reasoning deltas={reasoning_deltas}")
        print("answer:", answer)
        # An exact sequence check, because `"1" in answer` would pass on "10" alone.
        numbers = [int(token) for token in re.findall(r"\d+", answer)]
        print("counted 1 to 10:", numbers == list(range(1, 11)))

1

,

2

,

3

,

... (10 outputs left out)

9

,

10


status=completed  reasoning deltas=0
answer: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
counted 1 to 10: True


## 3. Multi-turn

The Responses API keeps no state between calls unless you ask it to. To hold a
conversation yourself, send it in `input`, including the model's output items as they
arrived. With `include=["reasoning.encrypted_content"]` the reasoning item carries an
encrypted blob that gives the next turn the model's earlier reasoning, which the model
card documents.

A turn sometimes arrives as a reasoning item with no message item at all, so send back
what `output` holds rather than assuming an answer is in it. Over five repeats of this
first turn on `us.xai.grok-4.6` in `us-east-1` on 20 October 2026 every one carried a
message item, as did eight on Grok 4.7. The one bare-reasoning turn seen that day was on
Grok 4.7 ([`03-grok-4-7`](03-grok-4-7.ipynb)), and the turn after it still recalled the
colour, because the conversation carries the reasoning item either way.

In [4]:
messages = [{"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."}]
first = client.responses.create(
    model=GROK, input=messages, include=["reasoning.encrypted_content"],
    max_output_tokens=2000, store=False,
)
messages += [item.model_dump(exclude_none=True) for item in first.output]  # reasoning and answer
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.responses.create(
    model=GROK, input=messages, include=["reasoning.encrypted_content"],
    max_output_tokens=2000, store=False,
)

# A turn can come back as a reasoning item with no message, and an empty answer printed
# on its own reads as a failed call, so say which it was.
print("sent back:", [item.type for item in first.output])
print("turn 1   :", answer_text(first) or "(no message item: this turn returned reasoning only)")
turn2_answer = answer_text(second)
print("turn 2   :", turn2_answer)
# Reasoning is on, so the one-word answer can carry a sentence after it: look for the
# word rather than comparing the whole string.
print("recalled teal:", "teal" in turn2_answer.lower())

sent back: ['reasoning', 'message']
turn 1   : noted
turn 2   : teal
recalled teal: True


With `store=True` the service keeps the response, and `previous_response_id` continues
from it, so the next request carries only the new turn. On `bedrock-runtime`, chaining
from a response made with `store=False` returns 400 `Previous response with id '…' not
found.`

`responses.retrieve` and `responses.delete` on `bedrock-runtime`'s `/openai/v1` answer
differently for different callers rather than at different times. In `us-east-1` on
2026-10-07 one identity got 200 from both over four stored responses on each of Grok 4.6
and Grok 4.7 at 07:03, 16 of 16 calls, and over five repeats at 07:39, 10 of 10, while a
second identity in the same account got 500 `internal_server_error` from both on 20 of 20
calls at 08:32, five repeats over `us.xai.grok-4.6` and five over `us.xai.grok-4.7`.
Waiting does not clear the 500, so read it as something about the caller rather than a
fault that passes, and look at the identity's permissions rather than retrying.
`bedrock-mantle` served both calls for both identities: 3 of 3 over `xai.grok-4.6` in
`us-west-2` for the refused one, and the section 11 run below for the other. The cell
stores one response, chains from it, then tries to delete it and reports whichever answer
it gets; its committed output comes from a run that got the 500 on the delete, in
`us-east-1` on 3 November 2026, where the `previous_response_id` turn above it in the
same cell still answered, so the 500 says something about reading back and deleting
rather than about storing. Prefer `store=False` and hold the conversation yourself, as
the cell above does; a response you do store ages out after 30 days.

In [5]:
turn1 = client.responses.create(
    model=GROK,
    input="My favourite colour is teal. Reply with one word: noted.",
    max_output_tokens=2000,
    store=True,  # kept for 30 days unless deleted, so turn 2 can refer to it
)
try:
    turn2 = client.responses.create(
        model=GROK,
        input="What is my favourite colour? One word.",
        previous_response_id=turn1.id,
        max_output_tokens=2000,
        store=False,
    )
    answer = answer_text(turn2)
    print("turn 2:", answer)
    print("recalled teal:", "teal" in answer.lower())
finally:
    # The delete runs even when the call above raises, because a response this cell
    # stored and did not delete is kept for 30 days. Both 200 and 500 came back from this
    # delete on 2026-10-07, so report whichever arrives rather than letting it stop the
    # notebook or replace the error on its way out.
    try:
        client.responses.delete(turn1.id)
        print("deleted the stored response")
    except openai.APIStatusError as exc:
        print(f"delete refused: {exc.status_code} {exc.code}; the response stays stored")

turn 2: teal
recalled teal: True


delete refused: 500 internal_server_error; the response stays stored


## 4. Reasoning

Grok 4.6 always reasons, and a response echoes the effort it used. The last line of the
cell below sends no `reasoning` field at all, which is the only way to see the default;
it came back `high` in October 2026 runs, on both endpoints. The `effort` in
`reasoning` can be `low`, `medium`, `high` or `xhigh`; `none` returned 400 `Unsupported
value` in September 2026 runs, so reasoning cannot be switched off. The reasoning is
encrypted: the `reasoning` item holds `encrypted_content` and no readable text.
`usage.output_tokens_details.reasoning_tokens` counts the reasoning you pay for.

In [6]:
response = client.responses.create(
    model=GROK,
    input='A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.',
    reasoning={"effort": "medium"},
    include=["reasoning.encrypted_content"],
    max_output_tokens=4000,
    store=False,
)
reasoning = [item for item in response.output if item.type == "reasoning"]
answer = answer_text(response)
print("answer   :", answer)
print("says $0.05:", says_five_cents(answer))  # the answer can run long, or word the amount
print("items    :", [item.type for item in response.output])
blob = "".join(item.encrypted_content or "" for item in reasoning)
readable = sum(len(item.content or []) + len(item.summary or []) for item in reasoning)
print(f"reasoning: {len(blob)} characters of encrypted_content, {readable} readable parts")
print("effort used:", response.reasoning.effort)  # echoed back by the service
print(f"reasoning_tokens={response.usage.output_tokens_details.reasoning_tokens}  status={response.status}")

# The default, which only a request with no `reasoning` field at all can show.
unset = client.responses.create(
    model=GROK, input="Reply with one word: noted.", max_output_tokens=2000, store=False
)
print("effort unset:", unset.reasoning.effort)

answer   : $0.05
says $0.05: True
items    : ['reasoning', 'message']
reasoning: 4366 characters of encrypted_content, 0 readable parts
effort used: medium
reasoning_tokens=211  status=completed


effort unset: high


## 5. Tool use

A Responses tool is flat: `name`, `description` and `parameters` sit at the top level,
where Chat Completions nests them under `function`. Each call arrives as a
`function_call` output item. Run it, send the call and a `function_call_output` item
with the same `call_id` back in `input`, and repeat until the model answers in text.
Check each call's arguments before running it: a call says the model wants a tool, and
says nothing about whether the arguments are usable.

In [7]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "name": "lookup_inventory",
    "description": "Current stock level for a SKU.",
    "parameters": {
        "type": "object",
        "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
        "required": ["sku"],
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t for t in TOOLS if t["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [8]:
items = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.responses.create(
        model=GROK, input=items, tools=TOOLS, max_output_tokens=2000, store=False
    )
    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        break
    for call in calls:
        args = parse_json_lenient(call.arguments)
        result = run_tool(call.name, args)
        print(f"tool call: {call.name}({args}) -> {result}")
        items.append({"type": "function_call", "call_id": call.call_id, "name": call.name,
                      "arguments": call.arguments})
        items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

answer = answer_text(response)
print("answer:", answer)
# 42 is the quantity the stand-in tool reports for A-100, so the answer must carry it.
# The loop breaks on any response with no function_call item, a reply cut off at the
# output cap included, so print status beside the verdict: "incomplete" marks an
# answer that the cap ended, which would otherwise read as a complete one.
print("42 units reported:", "42" in answer, " status:", response.status)

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: We have 42 units of SKU A-100 in stock.
42 units reported: True  status: completed


## 6. Structured output

`text.format` with a JSON Schema asks for output in that shape. Check `status` first,
because a cut-off answer is cut-off JSON, and parse leniently. A schema controls the
shape of the output and has no say over the values in it, so the cell checks each value
against the source text. The model card's feature table for the
`bedrock-runtime` endpoint lists structured outputs under `Not Supported`, where the
`bedrock-mantle` table lists them as supported (card read 20 October 2026); the request
below is on `bedrock-runtime` and returns the schema's shape with correct values, as it
did in September 2026 runs on both endpoints.

In [9]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.responses.create(
    model=GROK,
    input=("Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
           f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}"),
    text={"format": {"type": "json_schema", "name": "invoice", "strict": True, "schema": SCHEMA}},
    max_output_tokens=2000,
    store=False,
)
if response.status != "completed":
    raise RuntimeError(f"incomplete output ({getattr(response.incomplete_details, 'reason', response.status)})")

invoice = parse_json_lenient(response.output_text)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 7. Vision

Send the image as a data URL in an `input_image` part, before the text part. The slide
comes from a public AWS talk and ships with this repository, so the answer is known: its
title is `Ingestion from database`, and the cell counts how many of its three callout
lines the model quotes.

In [10]:
from bedrock import SLIDE_CALLOUTS, SLIDE_TITLE, keyword_recall, slide_data_url

response = client.responses.create(
    model=GROK,
    input=[{"role": "user", "content": [
        {"type": "input_image", "image_url": slide_data_url()},
        {"type": "input_text", "text": "Read this slide. Give its title, then quote the three green callout lines."},
    ]}],
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(response)
hits, total = keyword_recall(answer, SLIDE_CALLOUTS)
# Print the whole reply. The title check is a substring test, so a slice can cut off the
# words the verdict was computed from and leave a False with nothing to read it against.
print(answer)
# status sits beside the verdict because two different failures print the same 0/3: a
# reply the output cap ended before any text, and one that misread the slide. Reasoning
# is on at `high` by default here and spends this same budget (section 4).
print(f"\ntitle found: {SLIDE_TITLE.lower() in answer.lower()}   callouts found: {hits}/{total}"
      f"   status: {response.status}")

**Title:** Ingestion from database

**Green callouts:**
- ✓ Pay according to job duration
- ✓ Lower compute cost, up to 90%
- ✓ Lower storage cost for rarely accessed data

title found: True   callouts found: 3/3   status: completed


## 8. Chat Completions on the same path

Grok 4.6 also serves Chat Completions on the same path. The [model
card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-xai-grok-4-6.html)
says "The Chat Completions API does not return reasoning tokens" (read 22 October 2026),
and that holds for the reasoning text: the message carried no `reasoning` field at all on
3 of 3 calls at `reasoning_effort="high"` in `us-east-1` on 22 October 2026, and the cell
below defaults that field, so an absent one prints 0 characters. It does not hold for the
count. `usage.completion_tokens_details.reasoning_tokens` was non-zero on each of those
calls, 287, 257 and 267 against the 218 committed below, and 197 on one call that sent
no effort field. The count is the only thing here that tells you what the reasoning cost,
because there is no readable trace on either API (section 4).

The efforts this API accepts are not the ones the Responses API accepts: `minimal` is
taken here, 200 on 4 of 4 calls spending 145 to 178 reasoning tokens, and refused there
(Things that differ).

In [11]:
response = client.chat.completions.create(
    model=GROK,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_completion_tokens=4000,
)
message = response.choices[0].message
details = response.usage.completion_tokens_details
answer = (message.content or "").strip()
print("answer           :", answer)
print("says $0.05       :", says_five_cents(answer))
print("message.reasoning:", f"{len(getattr(message, 'reasoning', None) or '')} characters")
print("reasoning_tokens :", getattr(details, "reasoning_tokens", None))
print("finish_reason    :", response.choices[0].finish_reason)

answer           : $0.05
says $0.05       : True
message.reasoning: 0 characters
reasoning_tokens : 218
finish_reason    : stop


## 9. On `bedrock-runtime`: Converse

The same profile IDs work with Converse through boto3, which signs each request with
your AWS credentials. The reasoning arrives as a `reasoningContent` block holding
`redactedContent`, the Converse form of the encrypted trace, so read the text blocks by
type.

In [12]:
runtime = boto3.client("bedrock-runtime", region_name=REGION)
response = runtime.converse(
    modelId=GROK,
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
)
blocks = response["output"]["message"]["content"]
print("blocks    :", [next(iter(b)) for b in blocks])
print("reasoning :", [list(b["reasoningContent"]) for b in blocks if "reasoningContent" in b])
text = "".join(b["text"] for b in blocks if "text" in b)
answer = re.sub(r"\s*\\confidence\{\d+\}\s*$", "", text).strip()  # the same marker as answer_text
print("answer    :", answer)
print("says Canberra:", "canberra" in answer.lower())
print("stopReason:", response["stopReason"])

blocks    : ['reasoningContent', 'text']
reasoning : [['redactedContent']]
answer    : Canberra
says Canberra: True
stopReason: end_turn


## 10. On `bedrock-runtime`: InvokeModel

The card ticks a fourth API, Invoke, and its Sample Code section carries no example for
it (card read 2 November 2026), so the body shape below comes from measurement rather
than from the card. InvokeModel takes the Chat Completions body serialised into `body`,
rather than Converse's content blocks and `inferenceConfig`: `messages`, `max_tokens`
and `reasoning_effort` go in under the names Chat Completions uses (section 8), and the
reply is a `chat.completion` object read through `choices[0].message` and a
`finish_reason`, rather than an `output` message with a `stopReason`. Name the inference
profile here, as the other calls on this endpoint do; the bare `xai.grok-4.6` returns
`ValidationException: Invocation of model ID xai.grok-4.6 with on-demand throughput
isn’t supported.`

Where the reasoning goes is what changes on this route. There is no trace to read and no
`reasoning` field to look in: `message` carried `annotations`, `content`, `refusal` and
`role` and nothing else on 14 of 14 calls over both Grok profiles, while
`usage.completion_tokens_details.reasoning_tokens` counted the reasoning on each of
them. On `us.xai.grok-4.6` in `us-east-1` the one-word question below spent 94, 104 and
150 reasoning tokens over three calls at `high` on 2 November 2026, 106 on a call that
sent no effort field the same day, and 127 on the call committed below, which ran on 3
November 2026. Chat Completions behaves the same way on this family (section 8), and
other families do not: Nemotron returns its
working inline in `content` behind a closing tag
([`10-nvidia-nemotron/01`](../10-nvidia-nemotron/01-nemotron-nano-and-super.ipynb)).

Two things this route carries that Converse does not. It echoes `service_tier` back, and
it serves Flex and Priority: `flex` and `priority` each came back echoed on
`us.xai.grok-4.6` in `us-east-1` on 2 November 2026, on a path where Converse refuses
both (sections 9 and 14). The card also gives application inference profiles to the
Invoke and Converse APIs and not to Responses or Chat Completions, so this is the route
to take when you want inference cost tagged per application.

`invoke_model_with_response_stream` takes the same body. It answered on
`us.xai.grok-4.6` in `us-east-1` on 2 November 2026, delivering `chat.completion.chunk`
objects with the text in `choices[0].delta.content`. How many events carry a reply is
not fixed: a count from 1 to 5 arrived in 11 to 16 events over eight runs across the
two Grok profiles on 2 and 3 November 2026, so assemble the deltas rather than
counting events.
Section 2 streams this model on the Responses API, which is the path the rest of this
notebook uses.

In [13]:
CAPITAL = "What is the capital of Australia? Answer in one word."

# The Chat Completions body goes into `body` serialised, so the field names are the Chat
# Completions ones rather than Converse's inferenceConfig.
body = {
    "messages": [{"role": "user", "content": CAPITAL}],
    "max_tokens": 2000,
    "reasoning_effort": "high",
}
response = runtime.invoke_model(modelId=GROK, body=json.dumps(body))
payload = json.loads(response["body"].read())
choice = payload["choices"][0]

# Raise before reading the text. A budget that ran out leaves a short or empty `content`
# that prints here exactly like a finished answer, and the verdict below would read it as
# a wrong answer rather than as a call that never got to answer.
if choice["finish_reason"] != "stop":
    raise RuntimeError(f"finish_reason={choice['finish_reason']}, so the text is cut: "
                       "raise max_tokens")

content = choice["message"].get("content") or ""
answer = re.sub(r"\s*\\confidence\{\d+\}\s*$", "", content).strip()  # as answer_text does
print("object       :", payload["object"], "  service_tier:", payload.get("service_tier"))
print("message keys :", sorted(choice["message"]))  # no `reasoning` among them
print("reasoning    :",
      payload["usage"]["completion_tokens_details"]["reasoning_tokens"],
      "tokens counted, with no trace to read")
print("answer       :", answer)
print("says Canberra:", "canberra" in answer.lower())

object       : chat.completion   service_tier: default
message keys : ['annotations', 'content', 'refusal', 'role']
reasoning    : 127 tokens counted, with no trace to read
answer       : Canberra
says Canberra: True


## 11. On `bedrock-mantle`, and background mode

`bedrock-mantle` serves Grok 4.6 in `us-west-2` under the bare ID, and the card's
availability table also lists `us-gov-east-1` in GovCloud; in other commercial Regions it
returns 404 `The model 'xai.grok-4.6' does not exist` (measured in `us-east-1` on
18 October 2026). The GovCloud Region is not measured here: a token this repository's
test identity generates for it comes back 401 `The security token included in the request
is invalid.`, because the identity is in the commercial partition. `us-west-2` is also
where background mode runs: with `background=True` the call returns at once and the
result is fetched later, which suits long agent runs. `bedrock-runtime` refuses it with
400 `The background parameter is not supported.` A background response is stored, so the
cell deletes it once read.

In [14]:
import time

mantle = OpenAI(
    base_url="https://bedrock-mantle.us-west-2.api.aws/openai/v1",
    api_key=provide_token(region="us-west-2"),
    timeout=300,
)
job = mantle.responses.create(
    model="xai.grok-4.6",  # the bare ID: bedrock-mantle has no inference profiles
    input="Count from 1 to 10, separated by commas.",
    background=True,
    store=True,  # a background response is stored, so it can be fetched later
    max_output_tokens=2000,
)
print("submitted:", job.status)
try:
    for _ in range(60):  # poll for up to about two minutes
        if job.status not in ("queued", "in_progress"):
            break
        time.sleep(2)
        job = mantle.responses.retrieve(job.id)
    print("status   :", job.status)
    print("answer   :", answer_text(job))
finally:
    # Each step is guarded, so a refused cancel still reaches the delete and a refused
    # delete reports itself instead of replacing an error from the poll loop above.
    try:
        if job.status in ("queued", "in_progress"):
            mantle.responses.cancel(job.id)
    except openai.APIStatusError as exc:
        print(f"cancel refused: {exc.status_code} {exc.code}")
    try:
        mantle.responses.delete(job.id)  # otherwise it is kept for 30 days
        print("cleanup  : deleted the stored response")
    except openai.APIStatusError as exc:
        print(f"delete refused: {exc.status_code} {exc.code}; the response stays stored")

submitted: in_progress


status   : completed
answer   : 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
cleanup  : deleted the stored response


## 12. IAM permissions

Calling through an inference profile needs `bedrock:InvokeModel` on the profile and on
the foundation model in each Region the profile routes to. For Grok 4.6 on
`bedrock-runtime` the model card adds your account's default project,
`arn:aws:bedrock:{region}:{account-id}:project/default`. The cell builds that policy
from the profile's own ARNs and prints it with the account ID replaced.

In [15]:
account = boto3.client("sts").get_caller_identity()["Account"]
profile = bedrock.get_inference_profile(inferenceProfileIdentifier=GROK)

POLICY = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
        "Resource": [
            profile["inferenceProfileArn"],
            *(m["modelArn"] for m in profile["models"]),  # each Region the profile routes to
            f"arn:aws:bedrock:{REGION}:{account}:project/default",
        ],
    }],
}
safe_print(json.dumps(POLICY, indent=2))  # replaces the account ID before printing

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock:InvokeModel",
        "bedrock:InvokeModelWithResponseStream"
      ],
      "Resource": [
        "arn:aws:bedrock:us-east-1:123456789012:inference-profile/us.xai.grok-4.6",
        "arn:aws:bedrock:us-east-1::foundation-model/xai.grok-4.6",
        "arn:aws:bedrock:us-east-2::foundation-model/xai.grok-4.6",
        "arn:aws:bedrock:us-west-2::foundation-model/xai.grok-4.6",
        "arn:aws:bedrock:us-east-1:123456789012:project/default"
      ]
    }
  ]
}

## 13. Prompt caching

Caching on this family is implicit. No request parameter turns it on, and there are no
cache blocks to place: the service keys on the start of the request, so stable content
goes at the front and the question at the end.
`usage.input_tokens_details.cached_tokens` reports how much of the input was served from
cache, and `cache_write_tokens` stays 0, so filling the cache costs nothing extra, unlike
the explicit `cache_control` blocks Claude takes
([`02-anthropic-claude/02`](../02-anthropic-claude/02-thinking-tools-and-caching.ipynb)).
The model card prices a cache read at a quarter of the input rate, on each of its three
inference options (October 2026).

The cell sends one 18,000-token catalogue up to six times, asking a different question
each time, and stops at the first send that reports cached tokens. On `us.xai.grok-4.6`
in `us-east-1` on 10 October 2026 that was send 2, with 18,048 of the request's 18,164
input tokens served from cache. The run committed below also hit on send 2, on 3
November 2026, and reported the whole 18,175-token input cached rather than the prefix
alone, so how much of a request the cache covers is not fixed. Grok 4.3 on
`bedrock-mantle` does not hold a cache as reliably, which
[`01-grok-4-3`](01-grok-4-3.ipynb) measures.

In [16]:
import uuid

# Implicit caching keys on the start of the request, so the catalogue goes first and the
# question last. Nothing in the request asks for it. The run id makes the prefix new on
# every run, so send 1 cannot hit a cache an earlier run filled.
def entry(sku: int) -> tuple[int, float]:
    """Lead time in weeks and unit price, from values that do not run in a sequence a
    model could extrapolate instead of reading."""
    return sku * 13 % 7 + 2, (sku * 37) % 900 + 9.99

CATALOGUE = "\n".join(
    f"SKU A-{n} | lead time {entry(n)[0]} weeks | unit price ${entry(n)[1]:.2f}"
    for n in range(100, 1100)
)
PREFIX = f"Reference catalogue {uuid.uuid4()}\n{CATALOGUE}"
LOOKUPS = [(137, "unit price"), (242, "lead time"), (908, "unit price"),
           (451, "lead time"), (608, "unit price"), (771, "lead time")]

hit_on = None
for send, (sku, field) in enumerate(LOOKUPS, start=1):
    weeks, price = entry(sku)
    expected = f"{price:.2f}" if field == "unit price" else f"{weeks}"
    response = client.responses.create(
        model=GROK,
        input=f"{PREFIX}\n\nQuestion: What is the {field} of SKU A-{sku}? Answer with the value only.",
        reasoning={"effort": "low"},  # this model refuses "none" (section 4)
        max_output_tokens=500,
        store=False,
    )
    usage, details = response.usage, response.usage.input_tokens_details
    answer = answer_text(response)
    print(f"send {send}: A-{sku} {field}  input_tokens={usage.input_tokens} "
          f"cached={details.cached_tokens} written={details.cache_write_tokens} "
          f"status={response.status}")
    print(f"         answer={answer!r}, holds {expected}: {expected in answer}")
    if details.cached_tokens:
        hit_on = send
        break
print("first send served from cache:", hit_on or f"none of {len(LOOKUPS)}")

send 1: A-137 unit price  input_tokens=18175 cached=0 written=0 status=completed
         answer='$578.99', holds 578.99: True


send 2: A-242 lead time  input_tokens=18175 cached=18175 written=0 status=completed
         answer='5 weeks', holds 5: True
first send served from cache: 2


## 14. Service tiers

`service_tier` decides what a request costs and how it is scheduled. The card prices
Priority at 1.75 times the Standard per-token rate and Flex at half it, and its tier
table marks Standard, Priority and Flex supported and Reserved unsupported (card read 20
October 2026). Those display names are not all accepted values, so the first cell asks
for each one on the Responses API and prints the tier that served the request. On `us.xai.grok-4.6` in `us-east-1` on 19 October 2026, `auto` and
`default` were both served as `default`, as was omitting the field, `flex` and `priority`
echoed themselves, and `standard` and `reserved` were both refused with 400 under
different codes, `standard` by the service and `reserved` by the request parser. Chat
Completions on the same path takes the same field and the same values, and the cell sends
`flex` there rather than leaving it to be taken on trust.

Converse is a different field and a different answer, so the second cell measures it
separately. The field is `serviceTier={"type": ...}`, the enum botocore 1.43.59 will send
is `default`, `flex`, `priority` and `reserved` with no `auto` in it, and this model
serves Standard only: on the same profile, Region and date, `flex` and `priority` both
returned `ValidationException: The provided service tier is not supported for this
model.`, and `reserved`, which is in the enum, returned `ValidationException: The provided
model identifier is invalid.` rather than anything about the tier. Keep Flex and Priority
work on the Responses or Chat Completions path, or on Invoke, which served both on 2
November 2026 (section 10). A Converse response carries `serviceTier`
back only when the request asked for one; omitting the field left it absent in that run.

Read the tier back off the response rather than trusting the one you asked for, and read
it in the Region you deploy to. Tiers, quotas and latency across families are in
[`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb).

In [17]:
# The card's tier table has four columns, and the API takes a different set of values, so
# ask for each and print the tier that served the request rather than the one requested.
# The refusal code is recorded too, so a tier refused for some other reason cannot read as
# one of the refusals this notebook documents.
TIERS = ("auto", "default", "flex", "priority", "standard", "reserved")

served = {}
for tier in TIERS:
    try:
        reply = client.responses.create(
            model=GROK,
            input="Reply with one word: noted.",
            max_output_tokens=2000,
            service_tier=tier,
            store=False,
        )
        served[tier] = reply.service_tier
        print(f"{tier:9} served as {str(reply.service_tier):8} answer {answer_text(reply)!r}")
    except openai.BadRequestError as exc:
        served[tier] = f"refused:{exc.code}"
        print(f"{tier:9} refused   {exc.status_code} {exc.code} {exc.message}")

# Omitting the field is the other way to ask for Standard, and the card says so.
reply = client.responses.create(
    model=GROK, input="Reply with one word: noted.", max_output_tokens=2000, store=False)
served["omitted"] = reply.service_tier
print(f"{'omitted':9} served as {reply.service_tier}")

chat = client.chat.completions.create(  # the same field on Chat Completions
    model=GROK, messages=[{"role": "user", "content": "Reply with one word: noted."}],
    max_completion_tokens=2000, service_tier="flex")
served["chat flex"] = chat.service_tier
print(f"chat flex served as {chat.service_tier}  finish_reason {chat.choices[0].finish_reason}")

# Written out longhand from the card and the prose above rather than computed from the
# answers, so that a service that starts accepting standard or reserved shows up here.
expected = {"auto": "default", "default": "default", "flex": "flex", "priority": "priority",
            "standard": "refused:validation_error", "reserved": "refused:invalid_value",
            "omitted": "default", "chat flex": "flex"}
print("\nmatches what this notebook says:", served == expected)

auto      served as default  answer 'noted'


default   served as default  answer 'noted'


flex      served as flex     answer 'noted'


priority  served as priority answer 'noted'
standard  refused   400 validation_error Error code: 400 - {'error': {'message': 'service tier provided is invalid. Please check the documentation for valid service tiers', 'type': 'invalid_request_error', 'param': None, 'code': 'validation_error'}}


reserved  refused   400 invalid_value Error code: 400 - {'error': {'message': "Invalid value: 'reserved'. Supported values are: 'auto', 'default', 'flex', and 'priority'.", 'type': 'invalid_request_error', 'param': 'service_tier', 'code': 'invalid_value'}}


omitted   served as default


chat flex served as flex  finish_reason stop

matches what this notebook says: True


In [18]:
# Converse names the field differently and takes a different value set, and this model
# serves Standard only there, so the same question has to be asked of this path too.
converse_served = {}
for tier in ("default", "flex", "priority", "reserved"):
    try:
        reply = runtime.converse(
            modelId=GROK,
            messages=[{"role": "user", "content": [{"text": "Reply with one word: noted."}]}],
            inferenceConfig={"maxTokens": 2000},
            serviceTier={"type": tier},
        )
        converse_served[tier] = reply["serviceTier"]["type"]
        print(f"converse {tier:9} served as {reply['serviceTier']['type']:8} "
              f"stopReason {reply['stopReason']}")
    except runtime.exceptions.ValidationException as exc:
        converse_served[tier] = "refused"
        print(f"converse {tier:9} refused   {exc.response['Error']['Message']}")

# Omitting the field: Converse reports no tier at all rather than reporting the default.
plain = runtime.converse(
    modelId=GROK,
    messages=[{"role": "user", "content": [{"text": "Reply with one word: noted."}]}],
    inferenceConfig={"maxTokens": 2000},
)
print("converse omitted   serviceTier in response:", "serviceTier" in plain)

# botocore's own enum, which is where the absent `auto` shows without spending a call.
enum = runtime.meta.service_model.shape_for("ServiceTierType").enum
print("\nvalues Converse will send:", sorted(enum))
print("Converse serves Standard only:",
      converse_served == {"default": "default", "flex": "refused",
                          "priority": "refused", "reserved": "refused"}
      and "auto" not in enum)

converse default   served as default  stopReason end_turn
converse flex      refused   The provided service tier is not supported for this model.
converse priority  refused   The provided service tier is not supported for this model.


converse reserved  refused   The provided model identifier is invalid.


converse omitted   serviceTier in response: False

values Converse will send: ['default', 'flex', 'priority', 'reserved']
Converse serves Standard only: True


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Reading the answer | `output[0]` is a reasoning item on a reasoning call | Read `output_text`, or pick output items by `type`. |
| Stored by default | `store` defaults to true, and the response is kept for 30 days | Pass `store=False` unless you will chain from the response. Deleting can be refused here, as the next row says. |
| Reading back or deleting a stored response | `responses.retrieve` and `responses.delete` on `bedrock-runtime`'s `/openai/v1` follow the calling identity, not the clock: on 2026-10-07 in `us-east-1` one identity got 200 on 26 of 26 calls at 07:03 and 07:39, and a second got 500 `internal_server_error` on 20 of 20 at 08:32, over both Grok profiles, and runs on 18 and 19 October and 1, 2 and 3 November 2026 got the 500 on the delete with the `previous_response_id` turn in the same cell answering; `bedrock-mantle` served both identities | Prefer `store=False` and keep the conversation yourself (section 3), or store on `bedrock-mantle` (section 11). Where you do store, handle both answers, and look at the identity's permissions rather than retrying a 500. |
| Answer cut short | HTTP 200 with `status` `incomplete` and `incomplete_details.reason` `max_output_tokens` | Raise `max_output_tokens`, and check `status` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Prompt caching | Implicit: nothing in the request turns it on, `usage.input_tokens_details.cached_tokens` reports the hit, and `cache_write_tokens` stays 0. Send 2 of an 18,000-token prefix had 18,048 of 18,164 input tokens served from cache on 10 October 2026, and the whole 18,175 on 3 November 2026 (`us.xai.grok-4.6`, `us-east-1`) | Put stable content at the front of the request, and read `cached_tokens` to confirm the discount before you count on it (section 13). |
| Bare model ID on `bedrock-runtime` | 400 `Invocation of model ID xai.grok-4.6 with on-demand throughput isn’t supported` | Name `us.xai.grok-4.6` or `global.xai.grok-4.6`. |
| `us.` outside the US geo | 400 `The provided model identifier is invalid.` (measured from `eu-central-1`, 18 October 2026); from `ca-central-1` and `ca-west-1` the same ID answered 200, because the card lists both as source Regions for the US geo | Check the card's source-Region table for the geo you call, or call `GetInferenceProfile` from that Region as the card suggests, before falling back to `global.xai.grok-4.6`: `global.` routes to any commercial Region, so prefer `us.` where residency matters. |
| `bedrock-mantle` in another Region | 404 `The model 'xai.grok-4.6' does not exist` (`us-east-1`, 18 October 2026) | Call `bedrock-mantle` in `us-west-2` (section 11), or in `us-gov-east-1` if you work in GovCloud, which the card lists and this notebook does not measure. |
| Reasoning on Chat Completions | The model card says the Chat Completions API "does not return reasoning tokens". The message carries no `reasoning` field on 3 of 3 calls at `reasoning_effort="high"`, and `usage.completion_tokens_details.reasoning_tokens` is non-zero on those same calls: 287, 257 and 267, with 197 on one call that sent no effort field (`us-east-1`, 22 October 2026) | Read `reasoning_tokens` to see what the reasoning cost. There is no readable trace to show a user on either API (section 8). |
| Where the reasoning trace arrives | It goes by route, and two of the four routes carry no trace: `reasoning` items holding `encrypted_content` on Responses (section 4), a `reasoningContent` block holding `redactedContent` on Converse (section 9), and nothing readable on Chat Completions (section 8) or on Invoke, where `message` carried no `reasoning` field on 14 of 14 calls and the spend showed only in `usage.completion_tokens_details.reasoning_tokens` (`us-east-1`, 2 November 2026) | Read the trace by route, and where there is no trace, read `reasoning_tokens` for what it cost. A reader written for one route does not carry over to another (section 10). |
| Reasoning effort values | The two APIs take different sets, and the errors disagree about them. `none` and `max` are refused on both (3 of 3 calls each), and `minimal` is refused on the Responses API and accepted on Chat Completions, 200 on 4 of 4 calls spending 145 to 178 reasoning tokens. The messages do not agree on the set either: the Responses refusal of `minimal` lists `'none', 'low', 'medium', 'high', 'xhigh', and 'max'`, its refusal of `max` and of `none` lists `'low', 'medium', 'high', and 'xhigh'`, and on Chat Completions `max` returns a shorter 400 that lists nothing, `Unsupported value: 'max' is not supported with this model.` An unknown value is a different error on each, `Invalid value: 'bogus'.` with `minimal` in the list on Chat Completions and absent from it on Responses (`us-east-1`, 22 October 2026) | Use `low`, `medium`, `high` or `xhigh`, which both APIs accept, and do not take the supported-value list in the error as authoritative. |
| Default effort | A request with no `reasoning.effort` comes back with `reasoning.effort` `high`, on `us.xai.grok-4.6` and `global.xai.grok-4.6` in `us-east-1` and on `global.xai.grok-4.6` in `eu-central-1`, where the model card gives the default as `"low"` (October 2026) | Set `effort` explicitly when you care what you pay for reasoning. High effort is also what exhausts the budget at a high `temperature`, in the next row. |
| Sampling defaults and range | Not the OpenAI ones: a request that sends neither parameter comes back echoing `temperature` 0.7 and `top_p` 0.95, measured on `us.xai.grok-4.6` in `us-east-1` and on `global.xai.grok-4.6` in `us-east-1` and `eu-central-1` (October 2026). Sending one of the two leaves the other at the model's default. `temperature` above 2 returns 400 `Invalid 'temperature': decimal above maximum value. Expected a value <= 2, but got 3.0 instead.` and `top_p` above 1 the same error with a maximum of 1, and `temperature` 2.0 is inside the range and still returned no answer in eight trials of eight: seven with `status` `incomplete`, `incomplete_details.reason` `max_output_tokens` and the whole 2,000-token budget spent, and one with `status` `completed` and an empty `output_text` | Set both explicitly if your output depends on them, keep `temperature` at or below 1.5, which answered in four trials of four, and check that the text is non-empty as well as that `status` is `completed`. Code ported from another OpenAI-compatible model samples differently here (section 1). |
| Background mode | 400 `The background parameter is not supported.` on `bedrock-runtime` | Run background work on `bedrock-mantle` (section 11). |
| Web search | 400 ``Tool type 'web_search' is not supported for model `xai.grok-4.6`.`` | Search with your own tool and pass the results in. |
| Counting tokens | `CountTokens` returns `ValidationException: The provided model doesn't support counting tokens.` | Read token counts from `usage` on a real call. |
| Answer marker | An answer that ends in `\confidence{100}` | Remove it before display, as `answer_text` does. |
| Service tiers | On Responses and Chat Completions, `service_tier` takes `auto`, `default`, `flex` and `priority`; `flex` and `priority` echo themselves, and `auto`, `default` and omitting the field all come back `default`. A tier name on the card is not necessarily a value: `standard` returns 400 `service tier provided is invalid. Please check the documentation for valid service tiers`, and `reserved` returns 400 `Invalid value: 'reserved'. Supported values are: 'auto', 'default', 'flex', and 'priority'.` On Converse the field is `serviceTier={"type": ...}`, `auto` is not in the enum, and this model serves Standard only: `flex` and `priority` both return `ValidationException: The provided service tier is not supported for this model.` (`us.xai.grok-4.6`, `us-east-1`, 19 October 2026) | Send `flex` for work that can wait, which the card prices at half the Standard rate, or `priority` at 1.75 times it, read the tier back off the response, and do not pass a tier name taken from the card's prose. Keep that work on the Responses, Chat Completions or Invoke path, since Converse (section 9) refuses both tiers (sections 10 and 14, and [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Application inference profiles | Supported for the Invoke and Converse APIs, and not for Responses or Chat Completions, per the model card | Tag inference cost per application on the Invoke (section 10) or Converse (section 9) path. |
| Permissions | The profile alone is not enough on `bedrock-runtime`, per the model card | Allow the foundation models and your default project too (section 12). |

## Next

- Grok 4.7, on `bedrock-runtime` only: [`03-grok-4-7`](03-grok-4-7.ipynb)
- Grok 4.3, on `bedrock-mantle`: [`01-grok-4-3`](01-grok-4-3.ipynb)
- Converse and inference profiles in depth:
  [`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb)
- Other families on the Responses API:
  [`03-google-gemma/01`](../03-google-gemma/01-gemma4-end-to-end.ipynb) (Gemma 4),
  [`01-openai-gpt/01`](../01-openai-gpt/01-responses-api-core.ipynb) (OpenAI GPT)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)